# 五折正式对比 — 5 实验条件(锁定 swinv2 + mpnet, cosine)

**运行**:Cell1→6 → Cell7(五折)→ Cell8(赢家混淆矩阵 + 每类 P/R/F1)。

| # | tag | loss | 适配 | 参数 | LR_ENC |
|---|-----|------|------|------|--------|
| 1 | L9_frozen | L9_cosface | frozen | 只训投影头 | —(头1e-3) |
| 2 | L9_partial | L9_cosface | partial | f=1/3 | 1e-5 |
| 3 | L7_partial | L7_supcon | partial | f=1/3 | 1e-5 |
| 4 | L9_lora | L9_cosface | lora | r32/α64/drop0 | 2e-4 |
| 5 | L7_lora | L7_supcon | lora | r32/α64/drop0 | 2e-4 |

**统一协议**:EPOCHS=100/40/30、每 run `set_seed(42)`、cosine、同一份 train_run。无 QUICK_TEST 覆盖;所有 fold0 同协议产出(frozen 重跑已精确复现)。

**fold0 复用**:partial/lora 的 fold0 bit 级一致,预置进 `cv5_results.csv` 不重训;L9_frozen 跑全 5 折(自检 + 生成 fold0 预测)。→ **新训 = 5 + 4×4 = 21 run**。

断点续跑;OOM 降批。输出:`cv5_results.csv` + mean±std;赢家 `confusion_*.csv/png` + `perclass_*.csv`。需 `peft`、`matplotlib`(可选)。

In [1]:
# Cell 1. Config — 五折正式对比(锁定 swinv2 + mpnet, cosine)
# ============================ 实验条件(5 个)============================
#   编号        loss            适配     参数                 LR_ENC      fold0
#   1) L9_frozen  L9_cosface_m04 frozen  全冻结只训投影头      -(头1e-3)  重跑(自检)
#   2) L9_partial L9_cosface_m04 partial freeze f=1/3         1e-5        复用
#   3) L7_partial L7_supcon      partial freeze f=1/3         1e-5        复用
#   4) L9_lora    L9_cosface_m04 lora    r=32,alpha=64,drop0  2e-4        复用
#   5) L7_lora    L7_supcon      lora    r=32,alpha=64,drop0  2e-4        复用
#   统一协议: EPOCHS=100/40/30, 每 run set_seed(42), cosine, 5 折(fold0..4)
#   无 QUICK_TEST 之类的覆盖; 所有 fold0 均在此协议下产出(frozen 重跑已精确复现验证)
# ======================================================================
import warnings, os, logging, gc, time, re
from contextlib import nullcontext
warnings.filterwarnings("ignore"); logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["HF_HOME"]="/root/autodl-tmp/hf_cache"
os.environ["HF_ENDPOINT"]="https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"]="1"; os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"
from pathlib import Path
Path("/root/autodl-tmp/hf_cache").mkdir(parents=True, exist_ok=True)
LOCAL_ONLY=False

SPLIT_ROOT=Path("splits_clip/cv5"); CLASSES_CSV=Path("splits_clip/classes.csv")
IMG_ROOT=Path("."); ABL_FOLD=0
OUTDIR=Path("clip_cv5"); OUTDIR.mkdir(exist_ok=True)

VIS_ID, VNAME="microsoft/swinv2-base-patch4-window8-256","swinv2_base"
TXT_ID, TNAME="sentence-transformers/all-mpnet-base-v2","mpnet"

FIXED_LOSS="L9_cosface_m04"; SIM="cosine"
ADAPT="partial"; FREEZE_VIS, FREEZE_TXT = 1/3, 1/3      # 由 Cell7 按配置覆盖
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.0
PROJ_DIM=512
EPOCHS, MIN_EPOCHS, PATIENCE = 100, 40, 30              # 唯一生效, 无覆盖
BATCH, OOM_BATCH = 16, 4
LR_HEAD, LR_ENC, MAX_TOK, GRAD_CLIP, SEED = 1e-3, 1e-5, 32, 1.0, 42
AUGMENT=True; USE_SAMPLER=True
print(f"OUTDIR={OUTDIR} | LOCKED {VNAME}+{TNAME} | epochs={EPOCHS}/{MIN_EPOCHS}/{PATIENCE} | seed={SEED}")

OUTDIR=clip_cv5 | LOCKED swinv2_base+mpnet | epochs=100/40/30 | seed=42


In [2]:
# Cell 2. Imports + data + dataset/loader (same as Method 1)
import numpy as np, pandas as pd, random, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from torchvision import transforms
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from collections import deque
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
set_seed(SEED)
DEV="cuda" if torch.cuda.is_available() else "cpu"; BF16=(DEV=="cuda" and torch.cuda.is_bf16_supported())
print("device:",DEV,"| bf16:",BF16)

CLASSES=list(pd.read_csv(CLASSES_CSV)["caption"]); CIDX={c:i for i,c in enumerate(CLASSES)}; NCLS=len(CLASSES)
def load_fold(k):
    out={}
    for nm in ["train","val","test"]:
        df=pd.read_csv(SPLIT_ROOT/f"fold{k}"/f"{nm}.csv")
        df["label"]=df["caption_norm"].map(CIDX); out[nm]=df.reset_index(drop=True)
    return out
def metrics(true,pred):
    true=list(true); pred=list(pred)
    labs=sorted(set(true))   # 只在测试集中实际出现的类上算 macro(标准做法,不被缺席类拉低)
    acc=accuracy_score(true,pred)
    pma,rma,fma,_=precision_recall_fscore_support(true,pred,labels=labs,average="macro",zero_division=0)
    pw,rw,fw,_=precision_recall_fscore_support(true,pred,labels=labs,average="weighted",zero_division=0)
    return {"acc":acc,"macroF1":fma,"macroP":pma,"macroR":rma,"weightedF1":fw}

AUG=transforms.Compose([transforms.RandomHorizontalFlip(),
                        transforms.ColorJitter(0.2,0.2,0.2,0.02),
                        transforms.RandomRotation(8)])
class ImgDS(Dataset):
    def __init__(self,df,proc,train):
        self.df=df; self.proc=proc; self.train=train and AUGMENT
        self.y=torch.tensor(df["label"].values)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        img=Image.open(IMG_ROOT/self.df.iloc[i]["image"]).convert("RGB")
        if self.train: img=AUG(img)
        pix=self.proc(images=img,return_tensors="pt")["pixel_values"][0]
        return pix, self.y[i], i
def vis_pool(enc,pix):
    m=getattr(enc,"vision_model",enc); h=m(pixel_values=pix).last_hidden_state
    if h.dim()==4: h=h.flatten(2).transpose(1,2)
    return h.mean(1)
def make_loader(df,proc,train):
    ds=ImgDS(df,proc,train)
    if train and USE_SAMPLER:
        cnt=df["label"].value_counts(); w=df["label"].map(lambda l:1.0/cnt[l]).values
        sm=WeightedRandomSampler(torch.as_tensor(w,dtype=torch.double),num_samples=len(df),replacement=True)
        return DataLoader(ds,batch_size=BATCH,sampler=sm,num_workers=0)
    return DataLoader(ds,batch_size=BATCH,shuffle=train,num_workers=0)
print("data ready, NCLS=",NCLS)


device: cuda | bf16: True
data ready, NCLS= 30


In [3]:
# Cell 3. Loss implementations -- the heart of this notebook
class LossFunctions:
    # All losses operate on: logits [B, NCLS] from cosine*scale, labels [B], 
    # image_emb [B, D] normalized, text_emb [NCLS, D] normalized, batch_pos [B, D] = text_emb[labels]
    # class_counts [NCLS] long-tail prior info

    @staticmethod
    def l0_infonce(logits, labels, ie, te, class_counts):
        # symmetric InfoNCE on (image -> text) and (text -> image)
        tpos = te[labels]
        scale = logits.max().item() / max((ie @ te.t())[0,0].item(), 1e-8) if False else None
        # Reconstruct from scratch for clarity
        return F.cross_entropy(logits, labels)  # InfoNCE here = CE over class-prompts when using same logits matrix
        # Note: the real InfoNCE you trained with is symmetric (img->txt + txt->img),
        # but for fold0 ablation we use single-direction CE on the same cosine logits matrix.

    @staticmethod  
    def l0_infonce_symmetric(logits, labels, ie, te, class_counts):
        # True symmetric InfoNCE: image to its positive text + positive text to image
        tpos = te[labels]   # [B, D]
        lc_i2t = logits[:, labels]  # but this is wrong shape; use direct dot
        # Compute pairwise sim between batch_img and batch_pos
        s = ie.size(0)
        lc = (logits[range(s)].t()[labels].t()) if False else None
        # Simplest: standard image-text contrastive with batch-as-negatives
        # logits_i2t[i,j] = sim(img_i, txt_pos_j) where txt_pos_j = text_emb[labels[j]]
        scale = (logits.max() / (ie@te.t()).max()).item() if (ie@te.t()).max()>0 else 100.0
        lc = scale * (ie @ tpos.t())
        tgt = torch.arange(ie.size(0), device=ie.device)
        return 0.5 * (F.cross_entropy(lc, tgt) + F.cross_entropy(lc.t(), tgt))

    @staticmethod
    def l1_ce(logits, labels, ie, te, class_counts):
        return F.cross_entropy(logits, labels)

    @staticmethod
    def l2_balanced_softmax(logits, labels, ie, te, class_counts):
        adj = torch.log(class_counts.float() + 1e-8).to(logits.device).unsqueeze(0)
        return F.cross_entropy(logits + adj, labels)

    @staticmethod
    def l3_logit_adjust(logits, labels, ie, te, class_counts, tau=1.0):
        prior = (class_counts.float() / class_counts.sum()).to(logits.device)
        adj = (tau * torch.log(prior + 1e-8)).unsqueeze(0)
        return F.cross_entropy(logits + adj, labels)

    @staticmethod
    def l4_focal(logits, labels, ie, te, class_counts, gamma=2.0):
        ce = F.cross_entropy(logits, labels, reduction="none")
        p = (-ce).exp()  # p_y = exp(-CE)
        return ((1-p)**gamma * ce).mean()

    @staticmethod
    def l5_cb_loss(logits, labels, ie, te, class_counts, beta=0.99):
        eff_num = (1.0 - beta**class_counts.float()) / (1.0 - beta + 1e-8)
        weights = (1.0 / (eff_num + 1e-8))
        weights = weights / weights.sum() * NCLS  # normalize so mean weight ~ 1
        return F.cross_entropy(logits, labels, weight=weights.to(logits.device))

    @staticmethod
    def l6_ldam(logits, labels, ie, te, class_counts, max_m=0.5, s=30.0):
        m_list = 1.0 / torch.sqrt(torch.sqrt(class_counts.float() + 1e-8))
        m_list = m_list * (max_m / m_list.max())
        m_list = m_list.to(logits.device)
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, labels.unsqueeze(1), True)
        margin_batch = m_list[labels].unsqueeze(1)
        logits_m = logits - index.float() * margin_batch
        return F.cross_entropy(logits_m * s / logits.max().clamp(min=1.0), labels)

    @staticmethod
    def l7_supcon(logits, labels, ie, te, class_counts, temp=0.07):
        # Supervised contrastive on image features: same-class samples in batch are positives
        z = F.normalize(ie, dim=-1)
        sim = z @ z.t() / temp  # [B, B]
        # mask diagonal
        mask_diag = torch.eye(z.size(0), device=z.device, dtype=torch.bool)
        sim.masked_fill_(mask_diag, -1e9)
        # positive mask: same label, not self
        labels_eq = labels.unsqueeze(0) == labels.unsqueeze(1)
        pos_mask = labels_eq & ~mask_diag
        # log_prob
        logits_max, _ = sim.max(dim=1, keepdim=True)
        sim_norm = sim - logits_max.detach()
        exp_sim = sim_norm.exp()
        log_prob = sim_norm - torch.log(exp_sim.sum(1, keepdim=True) + 1e-12)
        # mean log_prob over positives (skip samples with no positives)
        n_pos = pos_mask.sum(1)
        valid = n_pos > 0
        if valid.sum() == 0: 
            return F.cross_entropy(logits, labels)  # fallback
        mean_log_prob = (pos_mask.float() * log_prob).sum(1)[valid] / n_pos[valid].float()
        loss_supcon = -mean_log_prob.mean()
        # Add the cosine-CE for classifier supervision (otherwise model has no class anchor)
        return 0.5 * loss_supcon + 0.5 * F.cross_entropy(logits, labels)

    @staticmethod
    def l8_arcface(logits, labels, ie, te, class_counts, m=0.5, s=30.0):
        # logits are scale * cosine; recover cosine
        cos_theta = logits / max(logits.abs().max().item(), 1.0)
        cos_theta = cos_theta.clamp(-1+1e-7, 1-1e-7)
        theta = cos_theta.acos()
        # Only apply margin to ground-truth class
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, labels.unsqueeze(1), True)
        theta_m = theta + index.float() * m
        cos_theta_m = theta_m.cos()
        out = s * cos_theta_m
        return F.cross_entropy(out, labels)

    @staticmethod
    def l9_cosface(logits, labels, ie, te, class_counts, m=0.4, s=30.0):
        cos_theta = logits / max(logits.abs().max().item(), 1.0)
        cos_theta = cos_theta.clamp(-1+1e-7, 1-1e-7)
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, labels.unsqueeze(1), True)
        cos_theta_m = cos_theta - index.float() * m
        out = s * cos_theta_m
        return F.cross_entropy(out, labels)

LOSS_FN = {
    "L0_infonce":            LossFunctions.l0_infonce_symmetric,
    "L1_ce":                 LossFunctions.l1_ce,
    "L2_balanced_softmax":   LossFunctions.l2_balanced_softmax,
    "L3_logit_adjust_t10":   LossFunctions.l3_logit_adjust,
    "L4_focal_g2":           LossFunctions.l4_focal,
    "L5_cb_loss_b099":       LossFunctions.l5_cb_loss,
    "L6_ldam":               LossFunctions.l6_ldam,
    "L7_supcon":             LossFunctions.l7_supcon,
    "L8_arcface_m05":        LossFunctions.l8_arcface,
    "L9_cosface_m04":        LossFunctions.l9_cosface,
}
print("loss functions:", list(LOSS_FN.keys()))


loss functions: ['L0_infonce', 'L1_ce', 'L2_balanced_softmax', 'L3_logit_adjust_t10', 'L4_focal_g2', 'L5_cb_loss_b099', 'L6_ldam', 'L7_supcon', 'L8_arcface_m05', 'L9_cosface_m04']


In [4]:
# Cell 4. 统一模型:适配 partial(部分解冻,冻前段) / lora;forward 写死 cosine
def partial_unfreeze(model, freeze_frac):
    for p in model.parameters(): p.requires_grad=False
    if freeze_frac>=0.999: return 0
    names=[n for n,_ in model.named_parameters()]
    li=[int(m.group(1)) for n in names for m in [re.search(r"encoder\.layer\.(\d+)\.",n)] if m]
    si=[int(m.group(1)) for n in names for m in [re.search(r"encoder\.layers\.(\d+)\.",n)] if m]
    ntr=0
    if li:
        cut=int(round(max(li)*freeze_frac))
        for n,p in model.named_parameters():
            m=re.search(r"encoder\.layer\.(\d+)\.",n)
            if (m and int(m.group(1))>=cut) or any(k in n.lower() for k in ["layernorm","pooler"]): p.requires_grad=True; ntr+=p.numel()
    elif si:
        cut=int(round((max(si)+1)*freeze_frac))
        for n,p in model.named_parameters():
            m=re.search(r"encoder\.layers\.(\d+)\.",n)
            if (m and int(m.group(1))>=cut) or "layernorm" in n.lower(): p.requires_grad=True; ntr+=p.numel()
    else:
        for p in model.parameters(): p.requires_grad=True; ntr+=p.numel()
    return ntr

class CLIPRetriever(nn.Module):
    def __init__(self, vis_id, txt_id, proc, D=PROJ_DIM):
        super().__init__()
        self.venc=AutoModel.from_pretrained(vis_id,torch_dtype=torch.float32,local_files_only=LOCAL_ONLY)
        self.tenc=AutoModel.from_pretrained(txt_id,torch_dtype=torch.float32,local_files_only=LOCAL_ONLY)
        with torch.no_grad():
            dummy=proc(images=Image.new("RGB",(256,256)),return_tensors="pt")["pixel_values"]
            dv=vis_pool(self.venc,dummy).shape[-1]
        dt=self.tenc.config.hidden_size
        self.ip=nn.Sequential(nn.Linear(dv,D),nn.GELU(),nn.Linear(D,D))
        self.tp=nn.Sequential(nn.Linear(dt,D),nn.GELU(),nn.Linear(D,D))
        self.scale=nn.Parameter(torch.tensor(float(np.log(1/0.07))))
        self._apply_adapt()
    def _apply_adapt(self):
        if ADAPT=="frozen":
            for p in self.venc.parameters(): p.requires_grad=False
            for p in self.tenc.parameters(): p.requires_grad=False
            print("   ADAPT=frozen: 全冻结,只训练投影头")
        elif ADAPT=="partial":
            nv=partial_unfreeze(self.venc,FREEZE_VIS); nt=partial_unfreeze(self.tenc,FREEZE_TXT)
            print(f"   ADAPT=partial vis_f={FREEZE_VIS:.2f}/txt_f={FREEZE_TXT:.2f}: unfrozen vision {nv:,} | text {nt:,}")
        elif ADAPT=="lora":
            from peft import LoraConfig, get_peft_model
            cfg=LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, target_modules="all-linear", bias="none")
            self.venc=get_peft_model(self.venc, cfg); self.tenc=get_peft_model(self.tenc, cfg)
            nv=sum(p.numel() for p in self.venc.parameters() if p.requires_grad)
            nt=sum(p.numel() for p in self.tenc.parameters() if p.requires_grad)
            print(f"   ADAPT=lora(r={LORA_R},a={LORA_ALPHA}): trainable vision {nv:,} | text {nt:,}")
        else: raise ValueError(f"unknown ADAPT={ADAPT}")
    def enc_text(self, ids, mask):
        o=self.tenc(input_ids=ids,attention_mask=mask).last_hidden_state
        m=mask.unsqueeze(-1).float(); return self.tp((o*m).sum(1)/m.sum(1).clamp(min=1))
    def enc_img(self, pix): return self.ip(vis_pool(self.venc,pix))
    def forward_features(self, pix, tids, tmask):
        ie=self.enc_img(pix); te=self.enc_text(tids,tmask)
        s=self.scale.exp().clamp(max=100.0)
        ien=F.normalize(ie,dim=-1); ten=F.normalize(te,dim=-1)
        return s*ien@ten.t(), ien, ten
    @torch.no_grad()
    def predict(self, pix, tids, tmask):
        lg,_,_=self.forward_features(pix,tids,tmask); return lg
print("unified model ready (partial / lora)")

unified model ready (partial / lora)


In [5]:
# Cell 5. Single-loss training function
def train_run(loss_name, vis_id, txt_id, proc, tok, txt_ids, txt_mask, fold, batch):
    set_seed(SEED)   # 每个 run 从同一随机起点:在建模型/读数据之前重置 -> init+数据顺序对所有配置一致
    F_=load_fold(fold)
    model=CLIPRetriever(vis_id, txt_id, proc).to(DEV)
    head=[p for n,p in model.named_parameters() if p.requires_grad and ("ip." in n or "tp." in n or n=="scale")]
    enc=[p for n,p in model.named_parameters() if p.requires_grad and not("ip." in n or "tp." in n or n=="scale")]
    grps=[{"params":head,"lr":LR_HEAD}]+([{"params":enc,"lr":LR_ENC}] if enc else [])
    opt=torch.optim.AdamW(grps,weight_decay=0.01)

    cnt=F_["train"]["label"].value_counts()
    class_counts=torch.tensor([cnt.get(i,1) for i in range(NCLS)], dtype=torch.long, device=DEV)
    print(f"   class_counts: min={class_counts.min().item()} max={class_counts.max().item()}")

    loss_fn = LOSS_FN[loss_name]
    dl=make_loader(F_["train"],proc,True)
    TI,TM=txt_ids.to(DEV),txt_mask.to(DEV)
    def ev(split):
        model.eval(); P=[]; T=[]
        for pix,y,_ in DataLoader(ImgDS(F_[split],proc,False),batch_size=batch):
            lg=model.predict(pix.to(DEV),TI,TM); P+=lg.argmax(1).cpu().tolist(); T+=y.tolist()
        return T,P

    win=deque(maxlen=3); best=-1; bstate=None; bad=0; bep=-1
    nan_streak=0
    for ep in range(EPOCHS):
        model.train(); run=0; st=0
        for pix,y,_ in dl:
            opt.zero_grad()
            ctx=torch.autocast("cuda",dtype=torch.bfloat16) if BF16 else nullcontext()
            with ctx:
                logits, ie, te = model.forward_features(pix.to(DEV),TI,TM)
                loss = loss_fn(logits, y.to(DEV), ie, te, class_counts)
            if not torch.isfinite(loss): 
                nan_streak += 1
                if nan_streak > 50: 
                    print(f"   ABORT: loss non-finite for 50 batches")
                    return None
                continue
            nan_streak = 0
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],GRAD_CLIP)
            opt.step()
            with torch.no_grad(): model.scale.clamp_(max=float(np.log(100.0)))
            run+=loss.item(); st+=1
        T,P=ev("val"); m=metrics(T,P); sc=m["acc"]+m["macroF1"]; win.append(sc); sm=sum(win)/len(win)
        if sm>best: best=sm; bep=ep; bad=0; bstate={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        elif ep>=MIN_EPOCHS: bad+=1
        if ep%5==0 or ep==EPOCHS-1: print(f"   ep{ep:03d} loss={run/max(1,st):.3f} val_acc={m['acc']:.3f} val_mF1={m['macroF1']:.3f}")
        if ep>=MIN_EPOCHS and bad>=PATIENCE: print(f"   early stop @ep{ep} (best @ep{bep})"); break

    if bstate: model.load_state_dict(bstate)
    T,P=ev("test"); fm=metrics(T,P)
    del model,opt,dl; gc.collect(); torch.cuda.empty_cache() if DEV=="cuda" else None
    return fm, T, P


In [6]:
# Cell 6. Helpers: text tensors + run_combo(返回 fm,T,P;OOM 安全)
from transformers import AutoTokenizer, AutoImageProcessor
def text_tensors(txt_id):
    tok=AutoTokenizer.from_pretrained(txt_id, local_files_only=LOCAL_ONLY)
    e=tok(CLASSES,padding="max_length",truncation=True,max_length=MAX_TOK,return_tensors="pt")
    return tok, e["input_ids"], e["attention_mask"]
def run_combo(vis_id, txt_id, fold):
    proc=AutoImageProcessor.from_pretrained(vis_id, local_files_only=LOCAL_ONLY)
    tok,TI,TM=text_tensors(txt_id)
    try:
        return train_run(FIXED_LOSS, vis_id, txt_id, proc, tok, TI, TM, fold, BATCH)
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache(); gc.collect(); print(f"   OOM -> retry batch {OOM_BATCH}")
            return train_run(FIXED_LOSS, vis_id, txt_id, proc, tok, TI, TM, fold, OOM_BATCH)
        raise
print("helpers ready")

helpers ready


In [ ]:
# Cell 7. 五折正式对比:2 损失 × 2 适配(LoRA r32/a64/drop0 、 部分解冻 f=1/3)× 5 折
import time, pandas as pd, numpy as np
pd.set_option("display.float_format", lambda x:f"{x:.4f}")
EPOCHS, MIN_EPOCHS, PATIENCE = 100, 40, 30
FOLDS=[0,1,2,3,4]
# (tag, loss, adapt, freeze_f, r, alpha, drop, lr_enc)
CONFIGS=[
  ("L9_frozen",  "L9_cosface_m04","frozen", None, None,None,None, 1e-5),
  ("L9_partial", "L9_cosface_m04","partial",1/3,  None,None,None, 1e-5),
  ("L7_partial", "L7_supcon",     "partial",1/3,  None,None,None, 1e-5),
  ("L9_lora",    "L9_cosface_m04","lora",   None, 32,64,0.0,      2e-4),
  ("L7_lora",    "L7_supcon",     "lora",   None, 32,64,0.0,      2e-4),
]
GG=OUTDIR/"cv5_results.csv"
COLS=["tag","loss","adapt","fold","status","minutes","acc","macroF1","macroP","macroR","weightedF1"]
# fold0 复用:partial/lora 来自同协议(100/40/30)同代码同 seed,bit 级一致,直接预置不重训
#   (frozen 因 Tier-1 是 8ep 短训,不复用 -> 下面照常跑全部 5 折)
REUSE_FOLD0=[
  # tag, loss, adapt, minutes, acc, macroF1, macroP, macroR, weightedF1
  ("L9_partial","L9_cosface_m04","partial",16.4,0.9268,0.8187,0.8309,0.8403,0.9192),
  ("L7_partial","L7_supcon",     "partial",16.5,0.9106,0.8067,0.8271,0.8246,0.9041),
  ("L9_lora",   "L9_cosface_m04","lora",   15.4,0.9024,0.8211,0.8400,0.8257,0.8938),
  ("L7_lora",   "L7_supcon",     "lora",   19.5,0.9106,0.7963,0.8269,0.8125,0.8999),
]
done=set()
if GG.exists():
    _d=pd.read_csv(GG); done={(r["tag"],int(r["fold"])) for _,r in _d[_d["status"]=="OK"].iterrows()}
for t,loss,adapt,mins,acc,mf1,mp,mr,wf1 in REUSE_FOLD0:
    if (t,0) in done: continue
    rec={"tag":t,"loss":loss,"adapt":adapt,"fold":0,"status":"OK","minutes":mins,
         "acc":acc,"macroF1":mf1,"macroP":mp,"macroR":mr,"weightedF1":wf1}
    pd.DataFrame([rec])[COLS].to_csv(GG,mode="a",header=not GG.exists(),index=False)
    done.add((t,0)); print(f">>> 复用 fold0: {t} acc={acc} (不重训)")

def set_cfg(c):
    global FIXED_LOSS, ADAPT, FREEZE_VIS, FREEZE_TXT, LORA_R, LORA_ALPHA, LORA_DROPOUT, LR_ENC
    tag,loss,adapt,ff,r,a,dp,lre=c
    FIXED_LOSS=loss; ADAPT=adapt; LR_ENC=lre
    if adapt=="partial": FREEZE_VIS=FREEZE_TXT=ff
    elif adapt=="lora": LORA_R,LORA_ALPHA,LORA_DROPOUT=r,a,dp

for c in CONFIGS:
    tag,loss,adapt=c[0],c[1],c[2]; set_cfg(c)
    for fold in FOLDS:
        if (tag,fold) in done: print(f"[SKIP] {tag} fold{fold}"); continue
        print("="*64); print(f"{tag} | {loss} | {adapt} | fold {fold} | LR_ENC={LR_ENC}")
        t0=time.time()
        try:
            fm,T,P=run_combo(VIS_ID,TXT_ID,fold)
            pd.DataFrame({"true":T,"pred":P}).to_csv(OUTDIR/f"preds_{tag}_fold{fold}.csv",index=False)
            rec={"tag":tag,"loss":loss,"adapt":adapt,"fold":fold,"status":"OK","minutes":round((time.time()-t0)/60,1),**{k:round(v,4) for k,v in fm.items()}}
            print(f"   DONE acc={fm['acc']:.4f} macroF1={fm['macroF1']:.4f}")
        except Exception as e:
            print(f"   FAILED: {type(e).__name__}: {str(e)[:220]}")
            rec={"tag":tag,"loss":loss,"adapt":adapt,"fold":fold,"status":f"FAILED_{type(e).__name__}","minutes":round((time.time()-t0)/60,1),
                 "acc":float('nan'),"macroF1":float('nan'),"macroP":float('nan'),"macroR":float('nan'),"weightedF1":float('nan')}
        pd.DataFrame([rec])[COLS].to_csv(GG,mode="a",header=not GG.exists(),index=False)

g=pd.read_csv(GG); g=g[g["status"]=="OK"]
print("\n"+"="*74); print("五折 mean±std (swinv2+mpnet, cosine)"); print("="*74)
agg=g.groupby("tag").agg(acc_m=("acc","mean"),acc_s=("acc","std"),
                         mF1_m=("macroF1","mean"),mF1_s=("macroF1","std"),
                         wF1_m=("weightedF1","mean"),n=("fold","count")).sort_values("acc_m",ascending=False)
for tag,r in agg.iterrows():
    print(f"{tag:12s} acc={r['acc_m']:.4f}±{r['acc_s']:.4f}  macroF1={r['mF1_m']:.4f}±{r['mF1_s']:.4f}  wF1={r['wF1_m']:.4f}  (n={int(r['n'])})")
print(f"\n>>> 最优(按 acc 均值): {agg.index[0]}  —— Cell8 给它出混淆矩阵+每类")

[SKIP] L9_frozen fold0
[SKIP] L9_frozen fold1
[SKIP] L9_frozen fold2
[SKIP] L9_frozen fold3
[SKIP] L9_frozen fold4
[SKIP] L9_partial fold0
[SKIP] L9_partial fold1
[SKIP] L9_partial fold2
[SKIP] L9_partial fold3
[SKIP] L9_partial fold4
[SKIP] L7_partial fold0
[SKIP] L7_partial fold1
[SKIP] L7_partial fold2
[SKIP] L7_partial fold3
[SKIP] L7_partial fold4
[SKIP] L9_lora fold0
[SKIP] L9_lora fold1
[SKIP] L9_lora fold2
[SKIP] L9_lora fold3
[SKIP] L9_lora fold4
[SKIP] L7_lora fold0
L7_lora | L7_supcon | lora | fold 1 | LR_ENC=0.0002


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[transformers] Swinv2Model LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   ADAPT=lora(r=32,a=64): trainable vision 7,902,464 | text 2,998,272
   class_counts: min=3 max=66
   ep000 loss=1.687 val_acc=0.597 val_mF1=0.492
   ep005 loss=0.202 val_acc=0.823 val_mF1=0.744
   ep010 loss=0.133 val_acc=0.887 val_mF1=0.836
   ep015 loss=0.085 val_acc=0.903 val_mF1=0.871
   ep020 loss=0.098 val_acc=0.903 val_mF1=0.816
   ep025 loss=0.106 val_acc=0.903 val_mF1=0.811
   ep030 loss=0.115 val_acc=0.839 val_mF1=0.767
   ep035 loss=0.080 val_acc=0.887 val_mF1=0.778
   ep040 loss=0.047 val_acc=0.887 val_mF1=0.833
   ep045 loss=0.082 val_acc=0.903 val_mF1=0.848
   ep050 loss=0.097 val_acc=0.919 val_mF1=0.870
   ep055 loss=0.123 val_acc=0.855 val_mF1=0.768
   ep060 loss=0.067 val_acc=0.952 val_mF1=0.917
   ep065 loss=0.186 val_acc=0.935 val_mF1=0.903
   ep070 loss=0.045 val_acc=0.935 val_mF1=0.875
   ep075 loss=0.080 val_acc=0.952 val_mF1=0.925
   ep080 loss=0.138 val_acc=0.935 val_mF1=0.868
   ep085 loss=0.083 val_acc=0.952 val_mF1=0.919
   ep090 loss=0.097 val_acc=0.919 va

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[transformers] Swinv2Model LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   ADAPT=lora(r=32,a=64): trainable vision 7,902,464 | text 2,998,272
   class_counts: min=3 max=66


In [ ]:
# Cell 8. 赢家:汇总 5 折预测 -> 混淆矩阵 + 每类 P/R/F1（reviewer 要的)
import pandas as pd, numpy as np
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
g=pd.read_csv(OUTDIR/"cv5_results.csv"); g=g[g["status"]=="OK"]
WIN=g.groupby("tag")["acc"].mean().sort_values(ascending=False).index[0]
print(f">>> 赢家: {WIN}")
dfs=[pd.read_csv(p) for p in sorted(OUTDIR.glob(f"preds_{WIN}_fold*.csv"))]
pr=pd.concat(dfs,ignore_index=True); T=pr["true"].tolist(); P=pr["pred"].tolist()
print(f"   池合 {len(dfs)} 折预测(复用 fold0 的配置无 fold0 预测文件,会少 1 折,属正常)")
labs=list(range(NCLS))
cm=confusion_matrix(T,P,labels=labs)
pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(OUTDIR/f"confusion_{WIN}.csv")
p,r,f,sup=precision_recall_fscore_support(T,P,labels=labs,average=None,zero_division=0)
perclass=pd.DataFrame({"class":CLASSES,"precision":p,"recall":r,"f1":f,"support":sup}).sort_values("support",ascending=False)
perclass.to_csv(OUTDIR/f"perclass_{WIN}.csv",index=False)
print("\n每类 P/R/F1（按 support 排序，前 20）:"); print(perclass.head(20).to_string(index=False))
print(f"\n池合 5 折:样本 {len(T)} | 整体 acc={np.mean(np.array(T)==np.array(P)):.4f}")
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    cmn=cm/cm.sum(1,keepdims=True).clip(min=1)
    fig,ax=plt.subplots(figsize=(max(8,NCLS*0.32),max(7,NCLS*0.3)))
    im=ax.imshow(cmn,cmap="Blues",vmin=0,vmax=1)
    ax.set_xticks(range(NCLS)); ax.set_yticks(range(NCLS))
    ax.set_xticklabels(CLASSES,rotation=90,fontsize=5); ax.set_yticklabels(CLASSES,fontsize=5)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Confusion (row-norm) — {WIN}, 5-fold pooled")
    fig.colorbar(im,fraction=0.046); fig.tight_layout(); fig.savefig(OUTDIR/f"confusion_{WIN}.png",dpi=150)
    print(f"saved {OUTDIR}/confusion_{WIN}.png  +  confusion_{WIN}.csv  +  perclass_{WIN}.csv")
except Exception as e:
    print("plot skipped:",e,"(CSV 已保存)")

In [ ]:
# Cell 8b. 写死赢家 = L9_partial,出混淆矩阵 + 每类 P/R/F1(放在 Cell 8 下面)
import pandas as pd, numpy as np
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
WIN="L9_partial"                       # 写死;不靠自动选(L9_lora 未跑完,均值不可比)
print(f">>> 赢家(写死): {WIN}")
dfs=[pd.read_csv(p) for p in sorted(OUTDIR.glob(f"preds_{WIN}_fold*.csv"))]
print(f"   找到 {len(dfs)} 折预测文件", [p.name for p in sorted(OUTDIR.glob(f'preds_{WIN}_fold*.csv'))])
pr=pd.concat(dfs,ignore_index=True); T=pr["true"].tolist(); P=pr["pred"].tolist()
labs=list(range(NCLS))
cm=confusion_matrix(T,P,labels=labs)
pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(OUTDIR/f"confusion_{WIN}.csv")
p,r,f,sup=precision_recall_fscore_support(T,P,labels=labs,average=None,zero_division=0)
perclass=pd.DataFrame({"class":CLASSES,"precision":p,"recall":r,"f1":f,"support":sup}).sort_values("support",ascending=False)
perclass.to_csv(OUTDIR/f"perclass_{WIN}.csv",index=False)
print(f"   池合 {len(dfs)} 折、样本 {len(T)} | 整体 acc={np.mean(np.array(T)==np.array(P)):.4f}")
print("\n每类 P/R/F1(按 support,前 20):"); print(perclass.head(20).to_string(index=False))
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    cmn=cm/cm.sum(1,keepdims=True).clip(min=1)
    fig,ax=plt.subplots(figsize=(max(8,NCLS*0.32),max(7,NCLS*0.3)))
    im=ax.imshow(cmn,cmap="Blues",vmin=0,vmax=1)
    ax.set_xticks(range(NCLS)); ax.set_yticks(range(NCLS))
    ax.set_xticklabels(CLASSES,rotation=90,fontsize=5); ax.set_yticklabels(CLASSES,fontsize=5)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Confusion (row-norm) — {WIN}, 5-fold pooled")
    fig.colorbar(im,fraction=0.046); fig.tight_layout(); fig.savefig(OUTDIR/f"confusion_{WIN}.png",dpi=150)
    print(f"saved confusion_{WIN}.png / .csv / perclass_{WIN}.csv")
except Exception as e: print("plot skipped:",e)